# LDS General Conference Scraper (Updated Dec 2024)

**✅ FIXED**: Now works with October 2024+ talks!

This notebook scrapes and cleans LDS General Conference talks from 1971-present.

## What's New:
- ✅ Fixed HTML parser for October 2024+ (website structure changed)
- ✅ Standardizes 102+ calling variations → 46 clean categories
- ✅ Parallel scraping for speed
- ✅ Automatic data cleaning

## Instructions:
1. Click **Runtime → Run all** in Google Colab
2. Wait 10-15 minutes for scraping to complete
3. Download `conference_talks_cleaned.csv` at the end

**Time**: ~10-15 minutes | **Output**: 4,000+ talks from 1971-2025

## Step 1: Install Libraries

In [ ]:
%pip install requests beautifulsoup4 pandas tqdm html5lib -q
print("✅ Libraries installed!")

## Step 2: Import Libraries

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import unicodedata
import time
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime

print(f"✅ Setup complete - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Step 3: Define Scraping Functions (Updated for Oct 2024+)

**KEY FIX**: The website changed its HTML structure in October 2024. Old talks use `<h1 id="title1">`, new talks use `<h1 id="random_id">`. This code handles both!

In [ ]:
def get_soup(url):
    try:
        r = requests.get(url, allow_redirects=True)
        r.raise_for_status()
        return BeautifulSoup(r.content, "html5lib")
    except:
        return None

def is_decade_page(url):
    return bool(re.search(r"/study/general-conference/\d{4}\d{4}", url))

def scrape_conference_pages(main_page_url):
    soup = get_soup(main_page_url)
    if not soup:
        return []
    all_links = []
    links = ["https://www.churchofjesuschrist.org" + a["href"]
             for a in soup.find_all("a", href=True)
             if re.search(r"/study/general-conference/(\d{4}/(04|10)|\d{4}\d{4})", a["href"])]
    for link in tqdm(links, desc="Getting conferences"):
        if is_decade_page(link):
            decade_soup = get_soup(link)
            if decade_soup:
                all_links.extend(["https://www.churchofjesuschrist.org" + a["href"]
                                 for a in decade_soup.find_all("a", href=True)
                                 if re.search(r"/study/general-conference/\d{4}/(04|10)", a["href"])])
        else:
            all_links.append(link)
    print(f"\n✅ Found {len(all_links)} conferences")
    return all_links

def scrape_talk_urls(conference_url):
    soup = get_soup(conference_url)
    if not soup:
        return []
    links = list(set(["https://www.churchofjesuschrist.org" + a["href"]
                     for a in soup.find_all("a", href=True)
                     if re.search(r"/study/general-conference/\d{4}/(04|10)/.*", a["href"])]))
    return [l for l in links if not l.endswith("session?lang=eng")]

def scrape_talk_data(url):
    """UPDATED: Handles both old and new HTML structures!"""
    try:
        soup = get_soup(url)
        if not soup:
            return {}
        # NEW: Try old structure first, fallback to new
        title_tag = soup.find("h1", {"id": "title1"}) or soup.find("h1")
        title = title_tag.text.strip() if title_tag else "No Title Found"
        speaker = (soup.find("p", {"class": "author-name"}) or {"text": "No Speaker Found"}).text.strip() if soup.find("p", {"class": "author-name"}) else "No Speaker Found"
        calling = (soup.find("p", {"class": "author-role"}) or {"text": "No Calling Found"}).text.strip() if soup.find("p", {"class": "author-role"}) else "No Calling Found"
        content_array = soup.find("div", {"class": "body-block"})
        content = "\n\n".join(p.text.strip() for p in content_array.find_all("p")) if content_array else "No Content Found"
        footnotes = "\n".join(f"{i}. {n.text.strip()}" for i, n in enumerate(soup.find_all("li", {"class": "study-note"}), 1)) or "No Footnotes Found"
        year = re.search(r'/(\d{4})/', url).group(1)
        season = "April" if "/04/" in url else "October"
        return {"title": title, "speaker": speaker, "calling": calling, "year": year,
                "season": season, "url": url, "talk": content, "footnotes": footnotes}
    except:
        return {}

def scrape_parallel(urls, workers=10):
    with ThreadPoolExecutor(max_workers=workers) as executor:
        results = list(tqdm(executor.map(scrape_talk_data, urls), total=len(urls), desc="Scraping talks"))
    return [r for r in results if r]

print("✅ Scraping functions ready!")